# *Klebsiella pneumoniae* genome QC workflow (300 assemblies)

Quality control of 300 *K. pneumoniae* assemblies (Animal, Clinical, Environmental; 100 each) before downstream analysis. 
Runs on Kaggle (CPU, no GPU).

**Pipeline**
1. Match genome files to Sample IDs and build a manifest (Steps 1-2).
2. Own FASTA statistics and exact-duplicate detection (Step 3).
3. Install and test CheckM2 and Kleborate (Steps 4-6).
4. Run Kleborate and CheckM2 in batches with checkpoints (Steps 7-8).
5. Apply the QC rule, make the final KEEP / EXCLUDE decision and write the report (Step 9).

**Tools:** CheckM2 1.1.0 (database from Zenodo record 14897628) and Kleborate 3.3.0b1. QC thresholds follow the KlebNET-GSP framework.

**Note on data:** the genome FASTA files and the Sample ID / assembly mapping spreadsheets belong to unpublished work and are not included. 
In the saved outputs, accession numbers, assembly names and the Kaggle account name are masked (`X` or a placeholder). The code is unchanged.

## Step 0: Kaggle default cell
Standard Kaggle starter cell. It only lists the files attached to the notebook (genome FASTA files and the mapping spreadsheets). File names in the saved output are masked.

In [1]:
# Step 0: list the input files attached to this Kaggle notebook (default Kaggle cell)
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/<kaggle-username>/kp-animal-100/GCF_XXXXXXXXX.X_ASMXXXXXXXvX_genomic.fna
/kaggle/input/datasets/<kaggle-username>/kp-animal-100/GCF_XXXXXXXXX.X_ASMXXXXXXXvX_genomic.fna
/kaggle/input/datasets/<kaggle-username>/kp-animal-100/GCA_XXXXXXXXX.X_ASMXXXXXXXvX_genomic.fna
/kaggle/input/datasets/<kaggle-username>/kp-animal-100/GCF_XXXXXXXXX.X_ASMXXXXXXXvX_genomic.fna
/kaggle/input/datasets/<kaggle-username>/kp-animal-100/GCA_XXXXXXXXX.X_ASMXXXXXXXvX_genomic.fna
/kaggle/input/datasets/<kaggle-username>/kp-animal-100/GCA_XXXXXXXXX.X_PDTXXXXXXXXX.X_genomic.fna
/kaggle/input/datasets/<kaggle-username>/kp-animal-100/GCA_XXXXXXXXX.X_ASMXXXXXXXvX_genomic.fna
/kaggle/input/datasets/<kaggle-username>/kp-animal-100/GCA_XXXXXXXXX.X_PDTXXXXXXXXX.X_genomic.fna
/kaggle/input/datasets/<kaggle-username>/kp-animal-100/GCA_XXXXXXXXX.X_PDTXXXXXXXXX.X_genomic.fna
/kaggle/input/datasets/<kaggle-username>/kp-animal-100/GCF_XXXXXXXXX.X_ASMXXXXXXXvX_genomic.fna
/kaggle/input/datasets/<kaggle-use

## Step 1: Setup, folders and helper functions
Imports, paths and small helpers used by every later step.
- `TMP` is wiped every session (tools, database, scratch). `WORK` is persisted (checkpoints, logs, results).
- `save_table` writes a `.part` file and then renames it, so a half-written file never looks finished.
- `run_stream` runs a command-line tool, prints only warnings/errors and keeps the full log.

In [2]:
# Step 1: setup (paths, helpers, logging)
import os, re, sys, time, json, shutil, hashlib, subprocess
from collections import deque
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import pandas as pd

T0    = time.time()
INPUT = Path("/kaggle/input")
TMP   = Path("/kaggle/temp")                    # wiped every session: tools, database, scratch
WORK  = Path("/kaggle/working/Kp_QC_Full")      # persisted: checkpoints, logs, results
RES, CK, LOGS = WORK / "results", WORK / "checkpoints", WORK / "logs"
for d in [TMP, RES, LOGS, CK / "checkm2", CK / "kleborate"]:
    d.mkdir(parents=True, exist_ok=True)

CONDA   = TMP / "miniforge3"
CM2, KB = CONDA / "envs" / "checkm2", CONDA / "envs" / "kleborate"
GROUPS  = ["Animal", "Clinical", "Environmental"]
THREADS = os.cpu_count() or 4

def find_one(name):
    hits = list(INPUT.rglob(name))
    assert len(hits) == 1, f"Expected exactly one '{name}' in /kaggle/input, found {len(hits)}"
    return hits[0]

def save_table(df, path, sep=","):
    """Write to a .part file, then rename: a half-written file can never look finished."""
    part = Path(str(path) + ".part")
    df.to_csv(part, index=False, sep=sep)
    os.replace(part, path)

def log_event(event, **info):
    with open(RES / "session_log.jsonl", "a") as fh:
        fh.write(json.dumps({"time": time.strftime("%Y-%m-%d %H:%M:%S"), "event": event, **info}) + "\n")

def tool_env(envdir):
    env = {**os.environ, "PATH": f"{envdir}/bin:{os.environ['PATH']}"}
    env.pop("PYTHONPATH", None)
    return env

SHOW = re.compile(r"rror|Traceback|Warning")

def run_stream(cmd, envdir, logfile):
    """Run a tool, print only problems, keep the full log, stop on error."""
    t, tail = time.time(), deque(maxlen=25)
    with open(logfile, "w") as log:
        p = subprocess.Popen([str(c) for c in cmd], env=tool_env(envdir), stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            if not line.strip():
                continue
            log.write(line); tail.append(line.rstrip()[:200])
            if SHOW.search(line):
                print("   ", line.rstrip()[:200], flush=True)
        rc = p.wait()
    if rc != 0:
        print("\n".join(tail))
    assert rc == 0, f"Command failed (exit code {rc}), see {logfile}"
    return time.time() - t

log_event("session_start", threads=THREADS)
print(f"Setup OK | threads: {THREADS} | results folder: {RES}")

Setup OK | threads: 4 | results folder: /kaggle/working/Kp_QC_Full/results


## Step 2: Match every genome file to its Sample ID (manifest)
Reads the three mapping spreadsheets (100 per source group) and finds the 300 `.fna` files. Assertions stop the run on empty cells, duplicate IDs or missing files. The result is `manifest`: one row per genome. GCA/GCF copies of the same assembly share an accession number, so they are flagged as pairs. Symlinks with short names (`KP001.fna`) make every tool report the Sample ID.

In [3]:
# Step 2: build the manifest (Sample_ID <-> assembly <-> file) and link genomes as KPxxx.fna
ACC = re.compile(r"(GC[AF]_\d+\.\d+)", re.I)
IN  = TMP / "Kp_QC_Full" / "input"

parts = []
for g in GROUPS:
    m = pd.read_excel(find_one(f"Assembly number {g}.xlsx"), dtype=str)
    m.columns = [c.strip() for c in m.columns]
    m = m[["Sample ID", "Assembly Number"]].rename(
        columns={"Sample ID": "Sample_ID", "Assembly Number": "Assembly_Number"})
    m = m.apply(lambda s: s.str.strip())
    m["Source_Group"] = g
    assert len(m) == 100, f"{g}: expected 100 mapping rows, found {len(m)}"
    parts.append(m)
mapping = pd.concat(parts, ignore_index=True)
assert not mapping.isna().any().any(), "Empty cells in the mapping files"
mapping["Key"] = mapping["Assembly_Number"].str.lower()
assert mapping["Sample_ID"].is_unique, "Duplicate Sample IDs"
assert mapping["Key"].is_unique, "Duplicate assembly numbers"
assert mapping["Sample_ID"].str.fullmatch(r"[A-Za-z0-9_-]+").all(), "Sample IDs must not contain dots or spaces"

genome_index = {}
for g in GROUPS:
    folder = next(p for p in INPUT.rglob(f"kp-{g.lower()}-100") if p.is_dir())
    files = sorted(folder.glob("*.fna"))
    assert len(files) == 100, f"{g}: expected 100 .fna files, found {len(files)}"
    for f in files:
        key = (g, ACC.search(f.name).group(1).lower())
        assert key not in genome_index, f"Duplicate genome accession {key}"
        genome_index[key] = f

mapping["Genome_Path"] = [genome_index.get((r.Source_Group, r.Key)) for r in mapping.itertuples()]
missing = mapping[mapping["Genome_Path"].isna()]
assert missing.empty, f"No genome file found for:\n{missing}"
assert mapping["Genome_Path"].nunique() == 300, "Two samples point to the same genome file"

manifest = mapping.drop(columns="Key").sort_values("Sample_ID").reset_index(drop=True)
manifest["Genome_File"] = [p.name for p in manifest["Genome_Path"]]
manifest["Genome_Path"] = manifest["Genome_Path"].astype(str)

# GCA (GenBank) and GCF (RefSeq) copies of one assembly share the same number
manifest["Acc_Number"] = manifest["Assembly_Number"].str.extract(r"_(\d+)\.")[0]
by_number = manifest.groupby("Acc_Number")["Sample_ID"].agg(list)
manifest["GCA_GCF_Pair"] = [", ".join(s for s in by_number[r.Acc_Number] if s != r.Sample_ID)
                            for r in manifest.itertuples()]

# short, dot-free names (KP001.fna) so every tool reports the Sample ID
shutil.rmtree(IN, ignore_errors=True)
IN.mkdir(parents=True)
for r in manifest.itertuples():
    (IN / f"{r.Sample_ID}.fna").symlink_to(r.Genome_Path)

ALL_IDS = manifest["Sample_ID"].tolist()
print(manifest.groupby("Source_Group").size().to_string())
print(f"\nAll {len(manifest)} genomes matched to a Sample ID and assembly number.")
print(f"Samples sharing an accession number with another sample (GCA/GCF): {(manifest.GCA_GCF_Pair != '').sum()}")

Source_Group
Animal           100
Clinical         100
Environmental    100

All 300 genomes matched to a Sample ID and assembly number.
Samples sharing an accession number with another sample (GCA/GCF): 20


## Step 3: Validate FASTA files, own assembly statistics, duplicate check
`fasta_stats` reads each file once and calculates contig count, genome size, N50, GC%, N bases and an MD5 fingerprint of the sequence only (headers ignored). It runs in parallel. Two genomes with the same MD5 are exact duplicates; the GenBank (GCA) copy is kept as the keep-candidate.

In [4]:
# Step 3: own FASTA statistics + exact-duplicate detection (MD5 of sequence)
def fasta_stats(path):
    lens, cur, n_bases, gc, acgt, started = [], 0, 0, 0, 0, False
    h = hashlib.md5()                       # fingerprint of the sequence only (headers ignored)
    with open(path, "rb") as fh:
        for line in fh:
            if line.startswith(b">"):
                if started:
                    lens.append(cur)
                started, cur = True, 0
                h.update(b">")
            else:
                s = line.strip().upper()
                cur += len(s)
                n_bases += s.count(b"N")
                g_c = s.count(b"G") + s.count(b"C")
                gc += g_c
                acgt += g_c + s.count(b"A") + s.count(b"T")
                h.update(s)
    if started:
        lens.append(cur)
    total = sum(lens)
    half, run, n50 = total / 2, 0, 0
    for L in sorted(lens, reverse=True):
        run += L
        if run >= half:
            n50 = L
            break
    return {"Contigs": len(lens), "Genome_Size_bp": total, "N_bases": n_bases,
            "Other_bases": total - acgt - n_bases, "N50_own": n50,
            "GC_percent": round(100 * gc / max(total - n_bases, 1), 2), "Seq_MD5": h.hexdigest()}

t = time.time()
with ProcessPoolExecutor(max_workers=THREADS) as ex:
    stats = list(ex.map(fasta_stats, manifest["Genome_Path"], chunksize=4))
assembly_stats = pd.concat([manifest[["Sample_ID"]], pd.DataFrame(stats)], axis=1)
bad = assembly_stats[(assembly_stats["Contigs"] == 0) | (assembly_stats["Genome_Size_bp"] == 0)]
assert bad.empty, f"Empty or invalid FASTA files:\n{bad}"
odd = assembly_stats[assembly_stats["Other_bases"] > 0]
print(f"Genome files validated: {len(assembly_stats)} readable FASTA files | "
      f"with N bases: {(assembly_stats.N_bases > 0).sum()} | with other non-ACGT characters: {len(odd)}")

# identical sequence = the same genome twice. Keep-candidate: the GCA copy, then the lowest Sample ID
order = manifest.assign(is_gcf=manifest["Assembly_Number"].str.lower().str.startswith("gcf"),
                        Seq_MD5=assembly_stats["Seq_MD5"]).sort_values(["is_gcf", "Sample_ID"])
first = order.groupby("Seq_MD5")["Sample_ID"].transform("first")
manifest["Duplicate_Of"] = first.where(first != order["Sample_ID"], "").reindex(manifest.index)

save_table(assembly_stats, RES / "assembly_stats_own.csv")
save_table(manifest, RES / "manifest_300.csv")

dups = manifest[manifest["Duplicate_Of"] != ""]
print(f"Counted {len(assembly_stats)} genomes in {time.time() - t:.0f}s")
print(f"Exact duplicate genomes (identical sequence): {len(dups)}  ->  unique genomes: {len(manifest) - len(dups)}")
display(dups[["Sample_ID", "Assembly_Number", "Source_Group", "Duplicate_Of"]])
unconfirmed = manifest[(manifest.GCA_GCF_Pair != "") & (manifest.Duplicate_Of == "")
                       & ~manifest.Sample_ID.isin(manifest.Duplicate_Of)]
print(f"Same accession number but NOT identical sequence: {len(unconfirmed)}")

Genome files validated: 300 readable FASTA files | with N bases: 53 | with other non-ACGT characters: 1
Counted 300 genomes in 23s
Exact duplicate genomes (identical sequence): 10  ->  unique genomes: 290


,Sample_ID,Assembly_Number,Source_Group,Duplicate_Of
81,KP082,gcf_XXXXXXXXX.X,Animal,KP025
85,KP086,gcf_XXXXXXXXX.X,Animal,KP028
86,KP087,gcf_XXXXXXXXX.X,Animal,KP038
88,KP089,gcf_XXXXXXXXX.X,Animal,KP047
89,KP090,gcf_XXXXXXXXX.X,Animal,KP048
91,KP092,gcf_XXXXXXXXX.X,Animal,KP050
95,KP096,gcf_XXXXXXXXX.X,Animal,KP060
278,KP279,gcf_XXXXXXXXX.X,Environmental,KP213
286,KP287,gcf_XXXXXXXXX.X,Environmental,KP241
287,KP288,gcf_XXXXXXXXX.X,Environmental,KP242


Same accession number but NOT identical sequence: 0


### Check: genomes with non-ACGT characters
Shows the genome(s) that contain characters other than A, C, G, T or N.

In [5]:
# Step 3b: list genomes with other (non-ACGT, non-N) characters
display(assembly_stats.loc[assembly_stats.Other_bases > 0,
        ["Sample_ID", "Contigs", "Genome_Size_bp", "N_bases", "Other_bases"]])

,Sample_ID,Contigs,Genome_Size_bp,N_bases,Other_bases
172,KP173,146,5649192,0,179


## Step 4: Install CheckM2 and Kleborate
Installs Miniforge and creates one conda environment per tool inside `/kaggle/temp`. It skips anything that is already installed. (The long conda log below is the saved output.)

In [6]:
%%bash
# Step 4: install conda environments (checkm2, kleborate)
set -euo pipefail
mkdir -p /kaggle/temp
CONDA_DIR=/kaggle/temp/miniforge3

if [ ! -x "$CONDA_DIR/bin/conda" ]; then
  cd /kaggle/temp
  wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O mf.sh
  bash mf.sh -b -p "$CONDA_DIR"
  rm -f mf.sh
fi
export PATH=$CONDA_DIR/bin:$PATH

[ -x $CONDA_DIR/envs/checkm2/bin/checkm2 ]     || conda create -y -q -n checkm2   -c conda-forge -c bioconda --override-channels checkm2=1.1.0
[ -x $CONDA_DIR/envs/kleborate/bin/kleborate ] || conda create -y -q -n kleborate -c conda-forge -c bioconda --override-channels kleborate

ls $CONDA_DIR/envs

PREFIX=/kaggle/temp/miniforge3
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.7.22-hbd8a1cb_0.conda
Extracting libgomp-16.2.0-he0feb66_4.conda
Extracting libzlib-1.3.2-h25fd6f3_3.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_2.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.14-9_cp314.conda
Extracting tzdata-2026c-h151e31d_0.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_7.conda
Extracting ld_impl_linux-64-2.46.1-default_hbd61a6d_102.conda
Extracting libgcc-16.2.0-ha9f2e26_4.conda
Extracting bzip2-1.0.8-hda65f42_10.conda
Extracting c-ares-1.34.8-hebe6cf0_2.conda
Extracting keyutils-1.6.3-h7cc23a3_1.conda
Extracting libev-4.33-h280c20c_3.conda
Extracting libexpat-2.8.1-hecca717_1.conda
Extracting libffi-3.7.0-h81df57d_1.conda
Extracting libiconv-1.18-h0cb94f2_3.conda
Extracting liblzma-5.8.3-hb03c661_1.conda
Extracting libmpdec-4.0.0-hb03c661_2.conda
Extracting libstdcxx-16.2.0-h934c35e_4.cond

warning  libmamba Security Warning: This transaction includes executing package scripts (pre/post-link/unlink) if present. These scripts can contain arbitrary code. Please ensure you trust the package sources.
WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html

WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html



## Step 5: Download the CheckM2 database
Downloads the reference database (about 3 GB) with up to 3 retries and checks that the file exists.

In [7]:
%%bash
# Step 5: download the CheckM2 database (3 attempts)
set -euo pipefail
export PATH=/kaggle/temp/miniforge3/envs/checkm2/bin:$PATH
mkdir -p /kaggle/temp/db

DBFILE=$(find /kaggle/temp/db -name "uniref100.KO.1.dmnd" | head -n1 || true)
if [ -z "$DBFILE" ]; then
  for i in 1 2 3; do
    checkm2 database --download --path /kaggle/temp/db --no_write_json_db && break || { echo "Download attempt $i failed, retrying in 30 s"; sleep 30; }
  done
fi

DBFILE=$(find /kaggle/temp/db -name "uniref100.KO.1.dmnd" | head -n1 || true)
[ -n "$DBFILE" ] || { echo "ERROR: database not found after 3 attempts"; exit 1; }
ls -lh "$DBFILE"

-rw-r--r-- 1 327701 4743 2.9G Feb  8  2025 /kaggle/temp/db/CheckM2_database/uniref100.KO.1.dmnd


[10/08/2026 11:06:25 AM] INFO: Command: Download database. Checking internal path information.
[10/08/2026 11:06:25 AM] INFO: The --no_write_json_db flag was set. Will download database but NOT modify internal database path information. You will need to provide database path in the future.
[10/08/2026 11:06:26 AM] INFO: Downloading https://zenodo.org/api/records/14897628/files/checkm2_database.tar.gz/content to /kaggle/temp/db/checkm2_database.tar.gz.
100%|██████████| 1.74G/1.74G [04:17<00:00, 6.73MiB/s]   
[10/08/2026 11:10:44 AM] INFO: Extracting files from archive...
[10/08/2026 11:11:27 AM] INFO: Verifying version and checksums...
[10/08/2026 11:11:38 AM] INFO: Verification success.
[10/08/2026 11:11:55 AM] INFO: Diamond DATABASE downloaded successfully! Consider running <checkm2 testrun> to verify everything works.


## Step 6: Test the tools
Records tool/database versions, then tests Kleborate on one genome. The species module was renamed between Kleborate releases, so the cell tries each known name until one works. It also saves the conda package lists for reproducibility.

In [12]:
# Step 6: check versions and find the working Kleborate species module
db_hits = list((TMP / "db").rglob("uniref100.KO.1.dmnd"))
assert db_hits, "CheckM2 database not found in /kaggle/temp/db - run the database install cell first"
DB = db_hits[0]
for exe in [CM2 / "bin" / "checkm2", KB / "bin" / "kleborate"]:
    assert exe.exists(), f"{exe} not found - run the install cell first"

def tool_out(cmd, envdir):
    p = subprocess.run([str(c) for c in cmd], capture_output=True, text=True, env=tool_env(envdir))
    return (p.stdout + "\n" + p.stderr).strip()

VERSIONS = {"CheckM2": tool_out(["checkm2", "--version"], CM2).splitlines()[-1].strip(),
            "Kleborate": tool_out(["kleborate", "--version"], KB).splitlines()[-1].strip(),
            "CheckM2_database": f"{DB.name}, {DB.stat().st_size:,} bytes"}

# Kleborate renamed its species module between releases: try each known name on one genome
KB_THREADS = ["--threads", THREADS] if "--threads" in tool_out(["kleborate", "--help"], KB) else []
KB_MODULES, probe_log = None, {}
for cand in ["enterobacterales__species", "general__species", "klebsiella__species"]:
    probe = TMP / "kb_probe"
    shutil.rmtree(probe, ignore_errors=True)
    mods = f"general__contig_stats,{cand}"
    probe_log[cand] = tool_out(["kleborate", "-m", mods, *KB_THREADS, "-o", probe,
                                "-a", IN / f"{ALL_IDS[0]}.fna"], KB)
    outs = sorted(probe.glob("*.txt")) if probe.exists() else []
    if outs:
        test = pd.concat([pd.read_csv(f, sep="\t", dtype=str) for f in outs], ignore_index=True)
        if len(test) == 1 and any(c == "species" or c.endswith("__species") for c in test.columns):
            KB_MODULES = mods
            print("Kleborate test on", ALL_IDS[0], "OK. Output files:", [f.name for f in outs])
            print("Columns:", list(test.columns))
            display(test)
            break
if KB_MODULES is None:
    print("=== kleborate --list_modules ===\n", tool_out(["kleborate", "--list_modules"], KB)[:3000])
    for cand, text in probe_log.items():
        print(f"\n=== tried {cand} ===\n", text[-1500:])
    raise AssertionError("No working Kleborate species module found - send me the text printed above")

(RES / "conda_env_checkm2.txt").write_text(tool_out([CONDA / "bin" / "conda", "list", "-p", CM2], CM2))
(RES / "conda_env_kleborate.txt").write_text(tool_out([CONDA / "bin" / "conda", "list", "-p", KB], KB))
print(json.dumps(VERSIONS, indent=1))
print("Kleborate modules used:", KB_MODULES)

Kleborate test on KP001 OK. Output files: ['general__contig_stats_output.txt']
Columns: ['strain', 'general__contig_stats__contig_count', 'general__contig_stats__N50', 'general__contig_stats__largest_contig', 'general__contig_stats__total_size', 'general__contig_stats__GC_content', 'general__contig_stats__ambiguous_bases', 'general__contig_stats__QC_warnings', 'general__species__species', 'general__species__species_match']


,strain,general__contig_stats__contig_count,general__contig_stats__N50,general__contig_stats__largest_contig,general__contig_stats__total_size,general__contig_stats__GC_content,general__contig_stats__ambiguous_bases,general__contig_stats__QC_warnings,general__species__species,general__species__species_match
0,KP001,9,5230931,5230931,5630899,57.16,no,-,Klebsiella pneumoniae,strong


{
 "CheckM2": "1.1.0",
 "Kleborate": "Kleborate v3.3.0b1",
 "CheckM2_database": "uniref100.KO.1.dmnd, 3,082,500,605 bytes"
}
Kleborate modules used: general__contig_stats,general__species


## Step 7: Run Kleborate (species + contig statistics)
`run_batches` runs a tool on genomes without a checkpoint, in batches (50 here). Each finished batch is saved as a `.tsv` checkpoint, so a disconnected session loses at most one batch and can resume.

In [13]:
# Step 7: Kleborate in batches of 50 with checkpoints
def done_ids(tool):
    """Sample IDs that already have a saved checkpoint for this tool."""
    files = sorted((CK / tool).glob("*.tsv"))
    return set(pd.concat([pd.read_csv(f, sep="\t", usecols=["Sample_ID"], dtype=str) for f in files])["Sample_ID"]) \
        if files else set()

def run_batches(tool, batch_size, make_cmd, find_reports, envdir):
    """Run a tool on every genome that has no checkpoint yet, in small batches.
    A disconnect loses at most the batch that was running."""
    todo = [s for s in ALL_IDS if s not in done_ids(tool)]
    print(f"{tool}: {len(ALL_IDS) - len(todo)} already done, {len(todo)} to run", flush=True)
    chunks = [todo[i:i + batch_size] for i in range(0, len(todo), batch_size)]
    t_start, n_done = time.time(), 0
    for chunk in chunks:
        name = f"{chunk[0]}_{chunk[-1]}"
        bdir = TMP / "run" / tool / name
        shutil.rmtree(bdir, ignore_errors=True)
        (bdir / "in").mkdir(parents=True)
        for s in chunk:
            (bdir / "in" / f"{s}.fna").symlink_to(IN / f"{s}.fna")
        secs = run_stream(make_cmd(bdir / "in", bdir / "out", chunk), envdir, LOGS / f"{tool}_{name}.log")

        reports = find_reports(bdir / "out")
        assert reports, f"{tool}: no report file found, see {LOGS / f'{tool}_{name}.log'}"
        tab = pd.concat([pd.read_csv(f, sep="\t", dtype=str) for f in reports], ignore_index=True)
        tab = tab.rename(columns={tab.columns[0]: "Sample_ID"})
        tab["Sample_ID"] = tab["Sample_ID"].str.replace(r"\.fna$", "", regex=True)
        tab = tab.groupby("Sample_ID", as_index=False).first()          # one row per genome
        assert set(tab["Sample_ID"]) <= set(chunk), f"{tool}: unexpected sample names {set(tab.Sample_ID) - set(chunk)}"
        no_result = [s for s in chunk if s not in set(tab["Sample_ID"])]
        tab = tab.set_index("Sample_ID").reindex(chunk).reset_index()   # genomes without a result stay as empty rows
        save_table(tab, CK / tool / f"{name}.tsv", sep="\t")
        shutil.rmtree(bdir, ignore_errors=True)

        n_done += len(chunk)
        left = (time.time() - t_start) / n_done * (len(todo) - n_done) / 60
        log_event("batch_done", tool=tool, batch=name, n=len(chunk), seconds=round(secs), no_result=no_result)
        print(f"  {name}: {len(chunk)} genomes in {secs / 60:.1f} min | {n_done}/{len(todo)} this session"
              f" | about {left:.0f} min left" + (f" | NO RESULT: {no_result}" if no_result else ""), flush=True)
    print(f"{tool}: all {len(done_ids(tool))} genomes have a checkpoint")

run_batches("kleborate", 50,
            lambda i, o, chunk: ["kleborate", "-m", KB_MODULES, *KB_THREADS, "-o", o,
                                 "-a", *[i / f"{s}.fna" for s in chunk]],
            lambda o: sorted(o.glob("*.txt")), KB)

kleborate: 0 already done, 300 to run
  KP001_KP050: 50 genomes in 1.5 min | 50/300 this session | about 8 min left
  KP051_KP100: 50 genomes in 1.5 min | 100/300 this session | about 6 min left
  KP101_KP150: 50 genomes in 1.6 min | 150/300 this session | about 5 min left
  KP151_KP200: 50 genomes in 1.5 min | 200/300 this session | about 3 min left
  KP201_KP250: 50 genomes in 1.7 min | 250/300 this session | about 2 min left
  KP251_KP300: 50 genomes in 1.5 min | 300/300 this session | about 0 min left
kleborate: all 300 genomes have a checkpoint


## Step 8: Run CheckM2 (completeness and contamination)
Same batch and checkpoint logic, batches of 25. This is the slowest step (about 35 min per batch).

In [14]:
# Step 8: CheckM2 in batches of 25 with checkpoints
run_batches("checkm2", 25,
            lambda i, o, chunk: ["checkm2", "predict", "--threads", THREADS, "-x", "fna", "--database_path", DB,
                                 "--input", i, "--output-directory", o, "--force"],
            lambda o: [f for f in [o / "quality_report.tsv"] if f.exists()], CM2)

checkm2: 0 already done, 300 to run
    /kaggle/temp/miniforge3/envs/checkm2/lib/python3.12/site-packages/checkm2/../sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using ve
  KP001_KP025: 25 genomes in 36.0 min | 25/300 this session | about 396 min left
    /kaggle/temp/miniforge3/envs/checkm2/lib/python3.12/site-packages/checkm2/../sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using ve
  KP026_KP050: 25 genomes in 35.2 min | 50/300 this session | about 356 min left
    /kaggle/temp/miniforge3/envs/checkm2/lib/python3.12/site-packages/checkm2/../sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using ve
  KP051_KP075: 25 genomes in 36.8 min | 75/300 this session | about 324 min left
    /kaggle/temp/miniforge3/envs/checkm2/lib/python3.12/site-packages/checkm2/../sklearn/base.py:525: Inco

## Step 9: Apply the QC rule and write the report
Thresholds are fixed in the first lines, before looking at results: completeness >= 90%, contamination < 5%, contigs <= 500, genome size 4.97-6.13 Mb, species = *Klebsiella pneumoniae* (Kleborate). Then:
1. merge the checkpoints with the own statistics (one row per sample);
2. cross-check contigs, size and N50 between own counts, CheckM2 and Kleborate;
3. apply the rule (PASS / FAIL, or PENDING if a tool has not finished);
4. decide per genome: `KEEP`, `EXCLUDE_DUPLICATE` or `EXCLUDE_QC_FAIL`;
5. add informational flags (do not change PASS/FAIL), build the flow, duplicate and metric tables;
6. write the CSV files, the Excel report and the methods text.

Assembly numbers in the saved tables are masked.

In [15]:
# Step 9: QC rule, cross-checks, final decision, tables and report files
# ---------------- thresholds (the QC rule, fixed before looking at results) ----------------
MIN_COMPLETENESS  = 90.0                              # >= 90 %
MAX_CONTAMINATION = 5.0                               # <  5 %
MAX_CONTIGS       = 500                               # <= 500
MIN_SIZE_BP, MAX_SIZE_BP = 4_970_000, 6_130_000       # 4.97 - 6.13 Mb
REQUIRED_SPECIES  = "Klebsiella pneumoniae"
REQUIRE_STRONG_SPECIES_MATCH = False                  # True = a "weak" Kleborate match also fails

def load_checkpoints(tool):
    files = sorted((CK / tool).glob("*.tsv"))
    if not files:
        return pd.DataFrame({"Sample_ID": pd.Series(dtype=str)})
    return pd.concat([pd.read_csv(f, sep="\t") for f in files], ignore_index=True).drop_duplicates("Sample_ID")

def pick(df, ending):
    """Find a Kleborate column by the end of its name, whatever the module prefix is."""
    hits = [c for c in df.columns if c == ending or c.endswith("__" + ending)]
    assert len(hits) == 1, f"Kleborate: expected one column ending in '{ending}', found {hits}. Columns: {list(df.columns)}"
    return hits[0]

import numpy as np

manifest       = pd.read_csv(RES / "manifest_300.csv", dtype=str).fillna("")
assembly_stats = pd.read_csv(RES / "assembly_stats_own.csv")
N = len(manifest)
assert manifest["Sample_ID"].is_unique and assembly_stats["Sample_ID"].is_unique, "Duplicate Sample IDs in saved tables"

cm2 = load_checkpoints("checkm2")
if len(cm2):
    cm2 = cm2[["Sample_ID", "Completeness", "Contamination", "Completeness_Model_Used", "Contig_N50",
               "Genome_Size", "Total_Contigs"]].rename(columns={
               "Contig_N50": "CM2_N50", "Genome_Size": "CM2_Size", "Total_Contigs": "CM2_Contigs"})
kb = load_checkpoints("kleborate")
if len(kb):
    kb = kb.rename(columns={pick(kb, "species"): "Kleborate_Species", pick(kb, "species_match"): "Species_Match",
                            pick(kb, "contig_count"): "KB_Contigs", pick(kb, "total_size"): "KB_Size",
                            pick(kb, "N50"): "KB_N50", pick(kb, "QC_warnings"): "KB_QC_Warnings"})
    kb = kb[["Sample_ID", "Kleborate_Species", "Species_Match", "KB_Contigs", "KB_Size", "KB_N50", "KB_QC_Warnings"]]

res = (manifest[["Sample_ID", "Assembly_Number", "Source_Group", "Genome_File", "Duplicate_Of", "GCA_GCF_Pair"]]
       .merge(assembly_stats, on="Sample_ID", validate="one_to_one")
       .merge(cm2, on="Sample_ID", how="left", validate="one_to_one")
       .merge(kb, on="Sample_ID", how="left", validate="one_to_one"))
for c in ["Completeness", "Contamination", "Completeness_Model_Used", "CM2_N50", "CM2_Size", "CM2_Contigs",
          "Kleborate_Species", "Species_Match", "KB_Contigs", "KB_Size", "KB_N50", "KB_QC_Warnings"]:
    if c not in res:
        res[c] = pd.NA

n_cm2, n_kb = len(cm2), len(kb)
COMPLETE = (n_cm2 == N) and (n_kb == N)

# ---------------- do the three independent counts agree? ----------------
num = lambda c: pd.to_numeric(res[c], errors="coerce")
pairs = {"contigs_vs_CheckM2": ("Contigs", "CM2_Contigs"), "contigs_vs_Kleborate": ("Contigs", "KB_Contigs"),
         "size_vs_CheckM2": ("Genome_Size_bp", "CM2_Size"), "size_vs_Kleborate": ("Genome_Size_bp", "KB_Size"),
         "N50_vs_CheckM2": ("N50_own", "CM2_N50"), "N50_vs_Kleborate": ("N50_own", "KB_N50")}
differs = pd.DataFrame({k: (num(a) != num(b)) & res[b].notna() for k, (a, b) in pairs.items()})
res["Crosscheck_OK"] = ~differs.any(axis=1)
res["Crosscheck_Detail"] = differs.apply(lambda r: ", ".join(k for k in differs.columns if r[k]), axis=1)

# ---------------- apply the QC rule ----------------
res["Pass_Completeness"]  = res["Completeness"] >= MIN_COMPLETENESS
res["Pass_Contamination"] = res["Contamination"] < MAX_CONTAMINATION
res["Pass_Contigs"]       = res["Contigs"] <= MAX_CONTIGS
res["Pass_GenomeSize"]    = res["Genome_Size_bp"].between(MIN_SIZE_BP, MAX_SIZE_BP)
res["Pass_Species"]       = res["Kleborate_Species"].eq(REQUIRED_SPECIES)
if REQUIRE_STRONG_SPECIES_MATCH:
    res["Pass_Species"] &= res["Species_Match"].eq("strong")
checks = {"Pass_Completeness": "completeness", "Pass_Contamination": "contamination",
          "Pass_Contigs": "contigs", "Pass_GenomeSize": "genome size", "Pass_Species": "species"}
res[list(checks)] = res[list(checks)].fillna(False).astype(bool)

def verdict(r):
    if all(r[k] for k in checks):
        return "PASS"
    measured = r["Pass_Contigs"] and r["Pass_GenomeSize"] and \
               (pd.isna(r["Completeness"]) or (r["Pass_Completeness"] and r["Pass_Contamination"])) and \
               (pd.isna(r["Kleborate_Species"]) or r["Pass_Species"])
    return "PENDING" if (measured and not COMPLETE) else "FAIL"
def reasons(r):
    out = []
    for k, v in checks.items():
        if not r[k]:
            miss = (k in ("Pass_Completeness", "Pass_Contamination") and pd.isna(r["Completeness"])) or \
                   (k == "Pass_Species" and pd.isna(r["Kleborate_Species"]))
            out.append(f"{v} (no result)" if miss else v)
    return ", ".join(dict.fromkeys(out))
res["Final_QC"] = res.apply(verdict, axis=1)
res["Fail_Reasons"] = res.apply(lambda r: "" if r["Final_QC"] == "PENDING" else reasons(r), axis=1)

# ---------------- duplicates and the final keep / exclude decision ----------------
# Order of exclusion: 1) duplicate copy of another sample, 2) fails the QC rule.
res["Is_Duplicate"] = res["Duplicate_Of"] != ""
res["Final_Decision"] = np.select(
    [res["Is_Duplicate"], res["Final_QC"] == "FAIL", res["Final_QC"] == "PENDING"],
    ["EXCLUDE_DUPLICATE", "EXCLUDE_QC_FAIL", "PENDING"], default="KEEP")
res["Exclusion_Reason"] = np.select(
    [res["Is_Duplicate"], res["Final_QC"] == "FAIL"],
    ["duplicate of " + res["Duplicate_Of"], res["Fail_Reasons"]], default="")

# ---------------- informational flags (they do NOT change PASS/FAIL) ----------------
def flags(r):
    f = []
    if r["N_bases"] > 0:                           f.append(f"ambiguous_bases({r['N_bases']})")
    if r["Other_bases"] > 0:                       f.append(f"other_non_ACGT({r['Other_bases']})")
    if pd.notna(r["Species_Match"]) and r["Species_Match"] != "strong": f.append("weak_species_match")
    if r["Completeness"] < 95:                     f.append("completeness<95")
    if r["Contamination"] >= 4:                    f.append("contamination>=4")
    if r["Contigs"] >= 400:                        f.append("contigs>=400")
    if abs(r["Genome_Size_bp"] - MIN_SIZE_BP) <= 0.01 * MIN_SIZE_BP or abs(r["Genome_Size_bp"] - MAX_SIZE_BP) <= 0.01 * MAX_SIZE_BP:
        f.append("size_within_1pct_of_limit")
    if pd.notna(r["Completeness_Model_Used"]) and "Specific" not in str(r["Completeness_Model_Used"]):
        f.append("checkm2_general_model")
    if pd.notna(r["KB_QC_Warnings"]) and str(r["KB_QC_Warnings"]) not in ("-", ""):
        f.append(f"kleborate:{r['KB_QC_Warnings']}")
    if not r["Crosscheck_OK"]:                     f.append(f"TOOL_MISMATCH({r['Crosscheck_Detail']})")
    return "; ".join(f)
res["QC_Flags"] = res.apply(flags, axis=1)
res["Genome_Size_Mb"] = (res["Genome_Size_bp"] / 1e6).round(3)

cols = ["Sample_ID", "Assembly_Number", "Source_Group", "Final_Decision", "Exclusion_Reason", "Final_QC", "Fail_Reasons",
        "Duplicate_Of", "Completeness", "Contamination", "Contigs", "Genome_Size_bp", "Genome_Size_Mb", "N50_own",
        "GC_percent", "N_bases", "Other_bases", "Kleborate_Species", "Species_Match",
        "Pass_Completeness", "Pass_Contamination", "Pass_Contigs", "Pass_GenomeSize", "Pass_Species",
        "QC_Flags", "Crosscheck_OK", "Completeness_Model_Used", "CM2_Contigs", "CM2_Size", "CM2_N50",
        "KB_Contigs", "KB_Size", "KB_N50", "KB_QC_Warnings", "GCA_GCF_Pair", "Seq_MD5", "Genome_File"]
final = res[cols].copy()
assert len(final) == N and final["Sample_ID"].is_unique, "The final table must have one row per sample"

# ---------------- duplicate pairs: kept copy next to removed copy ----------------
by_id = final.set_index("Sample_ID")
d = final[final["Duplicate_Of"] != ""]
k = by_id.loc[d["Duplicate_Of"]].reset_index()
dup_pairs = pd.DataFrame({
    "Source_Group": d["Source_Group"].values,
    "Kept_Sample": k["Sample_ID"].values, "Kept_Assembly": k["Assembly_Number"].values,
    "Removed_Sample": d["Sample_ID"].values, "Removed_Assembly": d["Assembly_Number"].values,
    "Same_Group": d["Source_Group"].values == k["Source_Group"].values,
    "Identical_Sequence": d["Seq_MD5"].values == k["Seq_MD5"].values,
    "Genome_Size_bp": k["Genome_Size_bp"].values, "Contigs": k["Contigs"].values,
    "Completeness_Kept": k["Completeness"].values, "Completeness_Removed": d["Completeness"].values,
    "Contamination_Kept": k["Contamination"].values, "Contamination_Removed": d["Contamination"].values,
    "Species_Kept": k["Kleborate_Species"].values, "Species_Removed": d["Kleborate_Species"].values,
    "Kept_Sample_QC": k["Final_QC"].values})
dup_pairs["Tool_Results_Identical"] = (
    (dup_pairs.Completeness_Kept == dup_pairs.Completeness_Removed) &
    (dup_pairs.Contamination_Kept == dup_pairs.Contamination_Removed) &
    (dup_pairs.Species_Kept == dup_pairs.Species_Removed))
assert dup_pairs["Identical_Sequence"].all(), "A duplicate pair does not have identical sequence"

# ---------------- sample flow: downloaded -> unique -> passed QC ----------------
grp = final.groupby("Source_Group")
flow = pd.DataFrame({
    "Assemblies_downloaded": grp.size(),
    "Duplicates_removed":    grp["Final_Decision"].apply(lambda s: (s == "EXCLUDE_DUPLICATE").sum()),
    "QC_failed":             grp["Final_Decision"].apply(lambda s: (s == "EXCLUDE_QC_FAIL").sum()),
    "Pending":               grp["Final_Decision"].apply(lambda s: (s == "PENDING").sum()),
    "Final_kept":            grp["Final_Decision"].apply(lambda s: (s == "KEEP").sum())})
flow.insert(2, "Unique_assemblies", flow["Assemblies_downloaded"] - flow["Duplicates_removed"])
flow.loc["All"] = flow.sum()
assert (flow["Unique_assemblies"] == flow[["QC_failed", "Pending", "Final_kept"]].sum(axis=1)).all(), "Flow table does not add up"

unique = final[final["Final_Decision"] != "EXCLUDE_DUPLICATE"]
kept   = final[final["Final_Decision"] == "KEEP"]
fail_by_rule = (~unique.loc[unique.Final_QC == "FAIL", list(checks)]).groupby(unique["Source_Group"]).sum()
fail_by_rule.columns = [checks[c] for c in fail_by_rule.columns]
fail_by_rule.loc["All"] = fail_by_rule.sum()

metrics = ["Completeness", "Contamination", "Contigs", "Genome_Size_Mb", "N50_own", "GC_percent"]
def metric_table(df):
    t = df.groupby("Source_Group")[metrics].agg(["median", "min", "max"]).round(2)
    t.columns = [f"{m}_{s}" for m, s in t.columns]
    return t
metrics_kept, metrics_unique = metric_table(kept), metric_table(unique)

thresholds = pd.DataFrame({
    "Criterion": ["Completeness (CheckM2)", "Contamination (CheckM2)", "Contig count", "Genome size",
                  "Species (Kleborate)", "Weak species match", "Duplicates"],
    "Rule": [f">= {MIN_COMPLETENESS:g} %", f"< {MAX_CONTAMINATION:g} %", f"<= {MAX_CONTIGS}",
             f"{MIN_SIZE_BP:,} - {MAX_SIZE_BP:,} bp", REQUIRED_SPECIES,
             "fails" if REQUIRE_STRONG_SPECIES_MATCH else "passes, flagged",
             "identical sequence (MD5 of contigs); GenBank (GCA) copy kept, the other removed"]})

# ---------------- methods text with the real numbers ----------------
n_mismatch = int((~final.Crosscheck_OK).sum())
n_dup, n_unique, n_keep = int((final.Duplicate_Of != "").sum()), len(unique), len(kept)
cross = ("agreed with CheckM2 and Kleborate for all assemblies" if n_mismatch == 0 else
         f"agreed with CheckM2 and Kleborate for {N - n_mismatch} of {N} assemblies")
methods = (
    f"A total of {N} assemblies were downloaded. Comparison of the sequences (MD5 checksum of the concatenated "
    f"contigs) showed that {n_dup} assemblies were exact duplicates of another assembly in the set (GenBank and "
    f"RefSeq copies of the same record); one copy of each was removed, leaving {n_unique} unique assemblies. "
    f"Genome quality was assessed using thresholds based on the KlebNET-GSP framework. Completeness and "
    f"contamination were estimated with CheckM2 ({VERSIONS['CheckM2']}; database: Zenodo record 14897628). "
    f"Assemblies were required to meet the following criteria: completeness >= {MIN_COMPLETENESS:.0f}% and "
    f"contamination < {MAX_CONTAMINATION:.0f}%; contig count <= {MAX_CONTIGS}; genome size "
    f"{MIN_SIZE_BP/1e6:.2f}-{MAX_SIZE_BP/1e6:.2f} Mb; and species confirmed as Klebsiella pneumoniae sensu stricto "
    f"by Kleborate ({VERSIONS['Kleborate']}; {KB_MODULES.split(',')[1]} module). Contig count, genome size and N50 "
    f"were calculated directly from the FASTA files and {cross}. Of the {n_unique} unique assemblies, {n_keep} met "
    f"all criteria and {n_unique - n_keep} were excluded, giving a final dataset of {n_keep} genomes ("
    + ", ".join(f"{g}: {int(flow.loc[g, 'Final_kept'])}" for g in GROUPS) + ").\n")

# ---------------- write the files ----------------
tag = "" if COMPLETE else "PARTIAL_"
save_table(final, RES / f"{tag}qc_results_all.csv")
if COMPLETE:
    save_table(kept, RES / "qc_final_kept_samples.csv")
    save_table(final[final.Final_Decision != "KEEP"], RES / "qc_excluded_samples.csv")
    save_table(dup_pairs, RES / "qc_duplicate_pairs.csv")
    save_table(final[final.QC_Flags != ""], RES / "qc_flagged_samples.csv")
    save_table(flow.reset_index(), RES / "qc_sample_flow_by_group.csv")
    save_table(metrics_kept.reset_index(), RES / "qc_metrics_by_group_kept.csv")
    (RES / "qc_methods_text.txt").write_text(methods)
    # one workbook with everything
    part = RES / "Kp_QC_report.part.xlsx"
    with pd.ExcelWriter(part) as xl:
        flow.reset_index().to_excel(xl, sheet_name="1_Sample_flow", index=False)
        final.to_excel(xl, sheet_name="2_All_300_results", index=False)
        kept.to_excel(xl, sheet_name="3_Final_kept", index=False)
        final[final.Final_Decision != "KEEP"].to_excel(xl, sheet_name="4_Excluded", index=False)
        dup_pairs.to_excel(xl, sheet_name="5_Duplicate_pairs", index=False)
        fail_by_rule.reset_index().to_excel(xl, sheet_name="6_QC_fail_by_rule", index=False)
        metrics_kept.reset_index().to_excel(xl, sheet_name="7_Metrics_kept", index=False)
        metrics_unique.reset_index().to_excel(xl, sheet_name="8_Metrics_all_unique", index=False)
        thresholds.to_excel(xl, sheet_name="9_Thresholds", index=False)
        pd.DataFrame({"Methods_text": [methods], **{k: [v] for k, v in VERSIONS.items()}}).T.reset_index() \
          .rename(columns={"index": "Item", 0: "Value"}).to_excel(xl, sheet_name="10_Methods_versions", index=False)
    os.replace(part, RES / "Kp_QC_report.xlsx")
    for old in ["PARTIAL_qc_results_all.csv", "qc_passed_samples.csv", "qc_failed_samples.csv",
                "qc_summary_by_group.csv", "qc_metrics_by_group.csv"]:
        (RES / old).unlink(missing_ok=True)

# ---------------- what you see on screen ----------------
pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_rows", 400)
print(f"CheckM2 results: {n_cm2}/{N} | Kleborate results: {n_kb}/{N} | "
      + ("COMPLETE - final files written" if COMPLETE else "PARTIAL - provisional table only, re-run the tool cells"))
print("\n=== 1. sample flow: downloaded -> unique -> final ===");          display(flow)
print("\n=== 2. duplicate pairs (the 'Removed' copy is excluded) ===")
display(dup_pairs[["Source_Group", "Kept_Sample", "Kept_Assembly", "Removed_Sample", "Removed_Assembly",
                   "Identical_Sequence", "Tool_Results_Identical", "Kept_Sample_QC"]])
print("\n=== 3. unique genomes failing each criterion (a genome can fail several) ===");  display(fail_by_rule)
print("\n=== 4. unique genomes excluded by QC ===")
display(unique.loc[unique.Final_QC == "FAIL", ["Sample_ID", "Assembly_Number", "Source_Group", "Completeness",
        "Contamination", "Contigs", "Genome_Size_bp", "Kleborate_Species", "Fail_Reasons"]])
print("\n=== 5. kept genomes with a flag (informational) ===")
display(kept.loc[kept.QC_Flags != "", ["Sample_ID", "Source_Group", "QC_Flags"]])
print("\n=== 6. metrics of the kept genomes, by group ===");                  display(metrics_kept.T)
print("\nGenomes where own counts / CheckM2 / Kleborate disagree:", n_mismatch)
if n_mismatch:
    display(res.loc[~res.Crosscheck_OK, ["Sample_ID", "Crosscheck_Detail", "Contigs", "CM2_Contigs", "KB_Contigs",
                                         "Genome_Size_bp", "CM2_Size", "KB_Size", "N50_own", "CM2_N50", "KB_N50"]])
if COMPLETE:
    print("\n" + methods)

CheckM2 results: 300/300 | Kleborate results: 300/300 | COMPLETE - final files written

=== 1. sample flow: downloaded -> unique -> final ===


,Assemblies_downloaded,Duplicates_removed,Unique_assemblies,QC_failed,Pending,Final_kept
Source_Group,,,,,,
Animal,100,7,93,6,0,87
Clinical,100,0,100,4,0,96
Environmental,100,3,97,7,0,90
All,300,10,290,17,0,273



=== 2. duplicate pairs (the 'Removed' copy is excluded) ===


,Source_Group,Kept_Sample,Kept_Assembly,Removed_Sample,Removed_Assembly,Identical_Sequence,Tool_Results_Identical,Kept_Sample_QC
0,Animal,KP025,gca_XXXXXXXXX.X,KP082,gcf_XXXXXXXXX.X,True,True,FAIL
1,Animal,KP028,gca_XXXXXXXXX.X,KP086,gcf_XXXXXXXXX.X,True,True,PASS
2,Animal,KP038,gca_XXXXXXXXX.X,KP087,gcf_XXXXXXXXX.X,True,True,PASS
3,Animal,KP047,gca_XXXXXXXXX.X,KP089,gcf_XXXXXXXXX.X,True,True,PASS
4,Animal,KP048,gca_XXXXXXXXX.X,KP090,gcf_XXXXXXXXX.X,True,True,PASS
5,Animal,KP050,gca_XXXXXXXXX.X,KP092,gcf_XXXXXXXXX.X,True,True,PASS
6,Animal,KP060,gca_XXXXXXXXX.X,KP096,gcf_XXXXXXXXX.X,True,True,PASS
7,Environmental,KP213,gca_XXXXXXXXX.X,KP279,gcf_XXXXXXXXX.X,True,True,PASS
8,Environmental,KP241,gca_XXXXXXXXX.X,KP287,gcf_XXXXXXXXX.X,True,True,PASS
9,Environmental,KP242,gca_XXXXXXXXX.X,KP288,gcf_XXXXXXXXX.X,True,True,PASS



=== 3. unique genomes failing each criterion (a genome can fail several) ===


,completeness,contamination,contigs,genome size,species
Source_Group,,,,,
Animal,0,1,4,1,0
Clinical,1,0,2,2,1
Environmental,1,3,1,4,2
All,2,4,7,7,3


... (3 outputs left out)

,Sample_ID,Source_Group,QC_Flags
1,KP002,Animal,ambiguous_bases(1263); kleborate:ambiguous_bases
10,KP011,Animal,contamination>=4
11,KP012,Animal,contamination>=4
26,KP027,Animal,kleborate:N50
27,KP028,Animal,kleborate:N50
28,KP029,Animal,ambiguous_bases(1200); kleborate:ambiguous_bases
29,KP030,Animal,ambiguous_bases(159); kleborate:ambiguous_bases
35,KP036,Animal,kleborate:N50
37,KP038,Animal,ambiguous_bases(700); kleborate:ambiguous_bases
38,KP039,Animal,ambiguous_bases(910); kleborate:ambiguous_bases



=== 6. metrics of the kept genomes, by group ===


Source_Group,Animal,Clinical,Environmental
Completeness_median,100.00,100.00,100.00
Completeness_min,99.99,99.99,91.20
Completeness_max,100.00,100.00,100.00
Contamination_median,0.51,0.34,0.29
Contamination_min,0.03,0.03,0.02
Contamination_max,4.82,4.96,4.33
Contigs_median,82.00,95.50,90.50
Contigs_min,2.00,2.00,1.00
Contigs_max,361.00,399.00,476.00
Genome_Size_Mb_median,5.53,5.64,5.54



Genomes where own counts / CheckM2 / Kleborate disagree: 0

A total of 300 assemblies were downloaded. Comparison of the sequences (MD5 checksum of the concatenated contigs) showed that 10 assemblies were exact duplicates of another assembly in the set (GenBank and RefSeq copies of the same record); one copy of each was removed, leaving 290 unique assemblies. Genome quality was assessed using thresholds based on the KlebNET-GSP framework. Completeness and contamination were estimated with CheckM2 (1.1.0; database: Zenodo record 14897628). Assemblies were required to meet the following criteria: completeness >= 90% and contamination < 5%; contig count <= 500; genome size 4.97-6.13 Mb; and species confirmed as Klebsiella pneumoniae sensu stricto by Kleborate (Kleborate v3.3.0b1; general__species module). Contig count, genome size and N50 were calculated directly from the FASTA files and agreed with CheckM2 and Kleborate for all assemblies. Of the 290 unique assemblies, 273 met all crite

## Step 10: Zip the results
Packages the results folder so it can be downloaded from Kaggle.

In [16]:
# Step 10: zip the results folder for download
import os, shutil
from IPython.display import FileLink

zip_path = shutil.make_archive("/kaggle/working/Kp_QC_Full_results", "zip", "/kaggle/working/Kp_QC_Full")
print(f"Zip ready: {os.path.getsize(zip_path) / 1e6:.1f} MB")
os.chdir("/kaggle/working")
FileLink("Kp_QC_Full_results.zip")

Zip ready: 0.2 MB


/kaggle/working/Kp_QC_Full_results.zip